**This example requires ipywidgets!**

# Kernel directive

A kernel directive is single line comment character sequence (`##@`) at the top of a block of code.

``` python
##@<execute mode> <option1_name=value>, <option2_name=value>....
# Only valid for this code block
```

The kernel directive only applies to the code block in which it is written and can modify how and where (namespace) the code is executed.
 
The first part of the directive is the execute mode, which is one of `task | thread | queue`, `queue` is default if the execute mode is omitted. Following the execute mode are options relevant to the execute mode. The `namespace_id` option is relevant to all execute modes.

- `##@task namespace_id=<value>`
- `##@thread thread_name=<value> namespace_id=<value>`
- `##@queue namespace_id=<value>` or `##@ namespace_id=<value>`

Lets define a function that we'll reuse for the remainder of the notebook.

In [ ]:
async def demo():
    import threading

    from ipywidgets import Button

    print(f"Thread name: '{threading.current_thread().name}'")
    button = Button(description="Finish")
    event = anyio.Event()
    thread_caller = Caller()  # Use the thread caller so the #@thread example works
    button.on_click(lambda _: thread_caller.call_no_context(event.set))
    display(button)
    await event.wait()
    button.close()
    print(f"Finished ... thread name: '{threading.current_thread().name}'")
    return "Finished"

Lets run it normally (queue)

In [ ]:
await demo()

Calling a code block without a kernel directive is equivalent to the kernel directive `##@queue namespace=`.

## Execute mode: task
``` python
#@task
...
```

The `task` mode instructs the kernel to execute the code in a task separate to the queue, Both `task` and `thread` execute modes can be started when the kernel is *busy executing*. There is no imposed limitation on the number of tasks (or threads) that can be run concurrently. 

In [ ]:
##@task
await demo()

## Execute mode: thread
``` python
#@thread, <thread_name=name>
...
```


In [ ]:
##@thread thread_name=my thread
await demo()

In [ ]:
##@thread thread_name=my thread
%threads # async kernel provides the `threads` magic

## option: namespace_id

Multiple namespaces are supported by async kernel. To use a different namespace, the namspace_id should be specified for each cell which uses the modified namespace. Failing to do so will use the default namespace (`namespace_id=''`).

In [ ]:
##@task namespace_id=my namespace

a = 10

assert a in shell.namespaces["my namespace"].values()
assert a not in shell.namespaces[""].values()

In [ ]:
##@task

list(shell.namespaces)

In [ ]:
##@task namespace_id=my namespace
a * 2

# Caller
Caller is a class provided to enable calling code in different threads and getting the result.

One caller instance is created per thread, and each of those instances can be retrieved by name using the `Caller.get_instance` class method. This method also creates a new thread if one doesn't already exist.

## Usage by the kernel

- Kernel uses the `Caller().call_soon` (method) - to execute cell code.
- Kernel uses the `Caller.to_thread_by_thread_name` (classmethod) to run code for the `##@thread` directive.

In [ ]:
import random
import threading
import time

import ipywidgets as ipw

outputs = {}


def job(n):
    thread = threading.current_thread()
    if not (out := outputs.get(thread)):
        outputs[thread] = out = ipw.HTML(description=thread.name)
        out.style.description_width = "initial"
        display(out)
    sleep_time = random.random() / 4
    out.value = f"started job {n} sleeping {sleep_time * 1000:03.0f} ms"
    time.sleep(sleep_time)
    return n


async def run_forever():
    n = 0
    while True:
        n += 1
        yield Caller.to_thread(job, n)


async for pr in Caller.as_completed(run_forever()):
    result = await pr.result()
    print(f"job finished: {result}", end="\r")